# 5. Confidence Intervals

A confidence interval provides a range of plausible values for a parameter. This notebook covers:
- CI for a **population mean** (t-interval)
- CI for a **proportion**
- **Bootstrap** confidence intervals
- Visualizing coverage and the effect of sample size on CI width

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline
np.random.seed(42)

## 5.1 CI for the Mean (t-interval) and for a Proportion

**Mean** (unknown $\sigma$): $\bar{x} \pm t_{\alpha/2, n-1} \cdot s/\sqrt{n}$

**Proportion** (Wald): $\hat{p} \pm z_{\alpha/2} \sqrt{\hat{p}(1-\hat{p})/n}$

In [ ]:
# CI for the mean
data = np.array([23.1, 25.4, 22.8, 24.6, 26.1, 23.9, 25.0, 24.2, 22.5, 25.8,
                 24.0, 23.5, 26.3, 24.7, 25.2])
n = len(data)
xbar, s = data.mean(), data.std(ddof=1)
ci_mean = stats.t.interval(0.95, df=n-1, loc=xbar, scale=s/np.sqrt(n))
print(f"Mean CI:  [{ci_mean[0]:.4f}, {ci_mean[1]:.4f}]  (n={n}, xbar={xbar:.2f})")

# CI for a proportion
x_success, n_total = 340, 500
p_hat = x_success / n_total
z_crit = stats.norm.ppf(0.975)
margin_p = z_crit * np.sqrt(p_hat * (1 - p_hat) / n_total)
print(f"Proportion CI: [{p_hat - margin_p:.4f}, {p_hat + margin_p:.4f}]  (p_hat={p_hat:.2f})")

## 5.2 Bootstrap Confidence Intervals

The **bootstrap** works for any statistic without distributional assumptions:
1. Draw $B$ resamples with replacement
2. Compute the statistic for each
3. Use the $\alpha/2$ and $1-\alpha/2$ quantiles as CI bounds

In [ ]:
def bootstrap_ci(data, stat_func=np.mean, B=10000, alpha=0.05):
    n = len(data)
    boot_stats = np.array([stat_func(np.random.choice(data, n, replace=True)) for _ in range(B)])
    lower = np.percentile(boot_stats, 100 * alpha/2)
    upper = np.percentile(boot_stats, 100 * (1 - alpha/2))
    return lower, upper, boot_stats

ci_lo, ci_hi, boot_means = bootstrap_ci(data)
ci_lo_med, ci_hi_med, boot_medians = bootstrap_ci(data, stat_func=np.median)
print(f"Bootstrap 95% CI for the mean:   [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"Bootstrap 95% CI for the median: [{ci_lo_med:.4f}, {ci_hi_med:.4f}]")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, boot, name, ci in zip(axes, [boot_means, boot_medians], ['Mean', 'Median'],
                                [(ci_lo, ci_hi), (ci_lo_med, ci_hi_med)]):
    ax.hist(boot, bins=50, edgecolor='black', alpha=0.7, color='steelblue', density=True)
    ax.axvline(ci[0], color='red', ls='--', lw=2, label=f'CI: [{ci[0]:.2f}, {ci[1]:.2f}]')
    ax.axvline(ci[1], color='red', ls='--', lw=2)
    ax.set_title(f'Bootstrap Distribution of {name}')
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5.3 Visualizing Coverage

If we repeat the experiment many times, approximately 95% of 95% CIs should contain the true parameter.

In [ ]:
true_mu, true_sigma = 50, 10
n_samples, n_experiments = 20, 50

fig, ax = plt.subplots(figsize=(10, 6))
covers = 0
for i in range(n_experiments):
    sample = np.random.normal(true_mu, true_sigma, n_samples)
    ci = stats.t.interval(0.95, df=n_samples-1, loc=sample.mean(),
                          scale=sample.std(ddof=1)/np.sqrt(n_samples))
    color = 'steelblue' if ci[0] <= true_mu <= ci[1] else 'red'
    if ci[0] <= true_mu <= ci[1]:
        covers += 1
    ax.plot([ci[0], ci[1]], [i, i], color=color, lw=1.5)
    ax.plot(sample.mean(), i, 'o', color=color, markersize=3)

ax.axvline(true_mu, color='black', ls='--', lw=2, label=f'True mean = {true_mu}')
ax.set_xlabel('Value')
ax.set_ylabel('Experiment')
ax.set_title(f'95% CI Coverage: {covers}/{n_experiments} = {covers/n_experiments:.0%}')
ax.legend()
plt.tight_layout()
plt.show()

## 5.4 Effect of Sample Size on CI Width

The CI width decreases as $1/\sqrt{n}$: to halve the width, quadruple the sample size.

In [ ]:
population = np.random.normal(100, 15, 10000)
sample_sizes = np.arange(10, 501, 10)
widths = []
for n in sample_sizes:
    sample = np.random.choice(population, size=n, replace=False)
    ci = stats.t.interval(0.95, df=n-1, loc=sample.mean(), scale=sample.std(ddof=1)/np.sqrt(n))
    widths.append(ci[1] - ci[0])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sample_sizes, widths, 'o-', markersize=3, color='steelblue')
ax.set_xlabel('Sample Size (n)')
ax.set_ylabel('CI Width')
ax.set_title('95% CI Width vs Sample Size')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

- A **95% CI** means: if we repeated sampling many times, ~95% of intervals would contain the true parameter
- Use **t-intervals** for the mean when $\sigma$ is unknown (the usual case)
- **Bootstrap CIs** require no distributional assumptions and work for any statistic
- CI width decreases with $\sqrt{n}$: to halve the width, quadruple the sample size
- Always report CIs alongside point estimates for transparent communication